# App rápida (sin reentrenar)
Usa resultados ya calculados (`resultados_para_colab.zip`, con el contenido de `data/processed`). Solo descarga los mapas del INE, corre el agente y levanta la app. Tarda ~10 min.

In [ ]:
!git clone -q https://github.com/AndreaHortaHernandez/ParticipacionCiudadanaINE.git
%cd ParticipacionCiudadanaINE
!pip -q install -r requirements.txt

Sube `resultados_para_colab.zip` cuando aparezca el botón **Elegir archivos**.

In [ ]:
from google.colab import files
import zipfile, os
subido = files.upload()
os.makedirs('data/processed', exist_ok=True)
for nombre in subido:
    zipfile.ZipFile(nombre).extractall('data/processed')
print(sorted(os.listdir('data/processed')))

In [ ]:
%%bash
set -e
mkdir -p data/geo && cd data/geo
for f in SECCION MUNICIPIO ENTIDAD; do curl -sL -A 'Mozilla/5.0' -O "https://pautas.ine.mx/transparencia/mapas/cob_2025/$f.zip" && mkdir -p $f && unzip -oq $f.zip -d $f && rm $f.zip; done
ls

In [ ]:
!python -W ignore -m src.geo
!python -m src.agent --anio 2027 --presupuesto-pct 10

### App
Abre la liga `https://….trycloudflare.com` que se imprime. Funciona mientras esta sesión siga abierta.

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
nohup ./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &
sleep 15
grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' cloudflared.log | head -1